# Imports

In [1]:
import os
import torch
import numpy as np
import open3d as o3d
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
from scene.dataset_readers import sceneLoadTypeCallbacks
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False

def prepare_output_and_logger(args):    
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])
        
    # Set up output folder
    print("Output folder: {}".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    # Create Tensorboard writer
    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# train.py
If you need more specific code review look at [tutorial_train_py.ipynb](tutorial_train_py.ipynb)

This tutorial follows the order in which the program starts after typing  
`python train.py -s your/file`  
on bash

## Parser Setting
First you set parser inside *train.py* before initializing training method using ArgumentParser

In [2]:
parser = ArgumentParser(description="Training script parameters")
lp = ModelParams(parser)
op = OptimizationParams(parser)
pp = PipelineParams(parser)
parser.add_argument('--ip', type=str, default="127.0.0.1")
parser.add_argument('--port', type=int, default=6009)
parser.add_argument('--debug_from', type=int, default=-1)
parser.add_argument('--detect_anomaly', action='store_true', default=False)
parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--quiet", action="store_true")
parser.add_argument('--disable_viewer', action='store_true', default=False)
parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
parser.add_argument("--start_checkpoint", type=str, default=None)

# In Jupyter, parse_known_args avoids runtime arguments such as -f that are injected by the notebook kernel.
args, _ = parser.parse_known_args([])
args.save_iterations.append(args.iterations)

# Convert the parser namespace into the lightweight argument objects used by the training code.
model_args = lp.extract(args)
opt_args = op.extract(args)
pipe_args = pp.extract(args)
model_args.source_path = os.path.join(model_args.source_path,"GaussianTest/Test2") 
# source_path is hardcoded on purpose for this tutorial, but you can change it to your own dataset path.

print("Loaded parser-backed arguments")
print("sh_degree:", model_args.sh_degree)
print("optimizer_type:", opt_args.optimizer_type)
print("source path: ", model_args.source_path)


Loaded parser-backed arguments
sh_degree: 3
optimizer_type: default
source path:  c:\Dev\gaussian-splatting-for-practice\GaussianTest/Test2


## `training` function initialize
```python
def training(model_args: ModelParams, opt_args: OptimizationParams, pipe_args: PipelineParams, testing_iterations: list, saving_iterations: list, checkpoint_iterations: list, checkpoint: str, debug_from: int):
```
After parser is set, inside train.py, `training` function is called, and the process below activates.  
This tutorial explains the code line by line so you can easily figure out what value is used or changed.  
Starts from `train.py` Line 48

In [3]:
first_iter = 0

When you use `prepare_output_and_logger` method, it creates new ouput folder with *unique_str* so you should check your output folder everytime after you run this code

In [4]:
tb_writer = prepare_output_and_logger(model_args)

Output folder: ./output/5f4909d2-3
Tensorboard not available: not logging progress


In [5]:
gaussians = GaussianModel(model_args.sh_degree, opt_args.optimizer_type)
scene = Scene(model_args, gaussians)
print("GaussianModel initialized successfully")

Reading camera 25/25
Loading Training Cameras


c:\Users\COM\anaconda3\envs\gaussian_splatting\lib\site-packages\torch\cuda\__init__.py:218: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Loading Test Cameras
Number of points at initialisation :  1768
GaussianModel initialized successfully


__These are ouputs from # here__  
Reading camera 25/25 # scene.dataset_readers.py Line 76  
Loading Training Cameras # scene.init.py Line 72  
Loading Test Cameras # scene.init.py Line 74    
Number of points at initialisation :  1768 # scene.gaussian_model.py Line 157 (create_from_pcd)

After you run the code above it creates `input.ply` on output folder from `prepare_output_and_logger` which has the same data from [GaussianTest/Test2/sparse/0](GaussianTest/Test2/sparse/0)/points3D.ply 

### Comparing *input.ply* and *points3D.ply*
If you need more detailed code review, look at [tutorial_train_py.ipynb](tutorial_train_py.ipynb) _Chapter Input_

In [31]:
ply_path = os.path.join(model_args.model_path, "input.ply")
ply_path2 = os.path.join(model_args.source_path, "sparse/0/points3D.ply")
pcd = o3d.io.read_point_cloud(ply_path)
pcd2= o3d.io.read_point_cloud(ply_path2)

pcd.paint_uniform_color([1.0, 0.0, 0.0]) # Red
pcd2.paint_uniform_color([0.0, 0.0, 1.0]) # Blue
pcd2.translate([1.0,0.0,0.0])

print(pcd)
print(pcd2)

points = np.asarray(pcd.points)
points2 = np.asarray(pcd2.points)
print(points)
print(points2)

o3d.visualization.draw_geometries([pcd,pcd2], window_name="PLY Comparison")

PointCloud with 1768 points.
PointCloud with 1768 points.
[[  4.34101534  -1.97590983   8.94548512]
 [  6.54583645  10.24704552   2.88442421]
 [  4.26116467  -0.91624063   9.97420979]
 ...
 [-14.37703228  10.57718468   7.03714371]
 [ -3.41339111   4.91355276  12.40224266]
 [-14.48631001  10.56289577   6.95793295]]
[[  5.34101534  -1.97590983   8.94548512]
 [  7.54583645  10.24704552   2.88442421]
 [  5.26116467  -0.91624063   9.97420979]
 ...
 [-13.37703228  10.57718468   7.03714371]
 [ -2.41339111   4.91355276  12.40224266]
 [-13.48631001  10.56289577   6.95793295]]


### create_from_pcd details
`create_from_pcd` function is called when *Scene* class object is made.  
For more details look at [tutorial_train_py](tutorial_train_py.ipynb) *Chapter create_from_pcd*

In [30]:
print(f"gaussians._exposure length: {len(gaussians._exposure)}")
print(f"gaussians._exposure: {gaussians._exposure}")

gaussians._exposure length: 25
gaussians._exposure: Parameter containing:
tensor([[[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.]],

        [[1.,

## Training Setup
This is where you setup training environment  
It sets the `gaussians.optimizer`, `gaussians.exposure_optimizer` with `torch.optim.Adam`.  
And sets `gaussians.xyz_scheduler_args`, `gaussians.exposrue_scheduler_args` with function `helper`

In [6]:
gaussians.training_setup(opt_args)

### Training Setup details

In [7]:
print(f"Optimizer Type: {gaussians.optimizer_type}","\n")
print(f"Optimizer: {gaussians.optimizer}", "\n")
print(f"Exposure_Optimizer: {gaussians.exposure_optimizer}" )

Optimizer Type: default 

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-15
    foreach: None
    fused: None
    lr: 0.000812694320678711
    maximize: False
    name: xyz
    weight_decay: 0

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-15
    foreach: None
    fused: None
    lr: 0.0025
    maximize: False
    name: f_dc
    weight_decay: 0

Parameter Group 2
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-15
    foreach: None
    fused: None
    lr: 0.000125
    maximize: False
    name: f_rest
    weight_decay: 0

Parameter Group 3
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-15
    foreach: None
    fused: None
    lr: 0.025
    maximize: False
    name: opacity
    weight_decay: 0

Parameter Group 4
   

In [83]:
print(f"gaussians.xyz_scheduler_args: {gaussians.xyz_scheduler_args}")
print(f"gaussians.exposure_scheduler_args: {gaussians.exposure_scheduler_args}")

gaussians.xyz_scheduler_args: <function get_expon_lr_func.<locals>.helper at 0x0000028FF7F0D040>
gaussians.exposure_scheduler_args: <function get_expon_lr_func.<locals>.helper at 0x0000028D7B4F1EE0>


`get_expon_lr_func` returns `helper` function,  
so when you print what variable `gaussians.xyz_scheduler_args`, `gaussians.exposure_scheduler_args` have, it prints address where helper function is located  

For more details, look at [tutorial_train_py](tutorial_train_py.ipynb) *Chapter Training Setup/get_expon_lr_func/How it works* 

__For line 53~55, since we don't use parameter *checkpoint* we will skip this part__
## Getting ready before starting training

In [8]:
bg_color = [1, 1, 1] if model_args.white_background else [0, 0, 0]
background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")
print(f"model_args.white_background: {model_args.white_background}")
print(f"bg_color: {bg_color}")
print(f"Background color: {background}")


model_args.white_background: False
bg_color: [0, 0, 0]
Background color: tensor([0., 0., 0.], device='cuda:0')


In [9]:
iter_start = torch.cuda.Event(enable_timing = True)
iter_end = torch.cuda.Event(enable_timing = True)

In [10]:
use_sparse_adam = opt_args.optimizer_type == "sparse_adam" and SPARSE_ADAM_AVAILABLE 
print(f"since {opt_args.optimizer_type} is false, use_sparse_adam is {use_sparse_adam}")

since default is false, use_sparse_adam is False


In [11]:
print(f"opt_args.depth_l1_weight_init: {opt_args.depth_l1_weight_init}, opt_args.depth_l1_weight_final: {opt_args.depth_l1_weight_final}, opt_args.iterations: {opt_args.iterations}")
depth_l1_weight = get_expon_lr_func(opt_args.depth_l1_weight_init, opt_args.depth_l1_weight_final, max_steps=opt_args.iterations)
print(f"depth_l1_weight: {depth_l1_weight}")

opt_args.depth_l1_weight_init: 1.0, opt_args.depth_l1_weight_final: 0.01, opt_args.iterations: 30000
depth_l1_weight: <function get_expon_lr_func.<locals>.helper at 0x0000027E7E4570D0>


In [35]:
viewpoint_stack = scene.getTrainCameras().copy()
viewpoint_indices = list(range(len(viewpoint_stack)))
print(f"viewpoint_stack: {viewpoint_stack}")
print(f"viewpoint_stack inside value: {viewpoint_stack[0].T}")
print(f"viewpoint_stack inside value: {viewpoint_stack[1].T}")
print(f"viewpoint_indices: {viewpoint_indices}")

viewpoint_stack: [Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera(), Camera()]
viewpoint_stack inside value: [-1.02276433 -3.34115737  2.40367095]
viewpoint_stack inside value: [ 0.89932992 -2.38217727  1.59730666]
viewpoint_indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [13]:
ema_loss_for_log = 0.0
ema_Ll1depth_for_log = 0.0

In [14]:
print(f"First iteration: {first_iter}, Last iteration: {opt_args.iterations}")
progress_bar = tqdm(range(first_iter, opt_args.iterations), desc="Training progress")

First iteration: 0, Last iteration: 30000


Training progress:   0%|          | 0/30000 [00:00<?, ?it/s]

In [15]:
first_iter += 1

## Start training  
It starts with the code 
```python
for iteration in range(first_iter, opt.iterations + 1):
```
But we want to see what happens inside the loop.  
And from line 74~87 the code is trying to connect to viewer so it doesn't matter whether this works  or not.  
So, we are going to start from train.py Line 89  

Let's assume that we are going through 1 loop of training.

In [16]:
iter_start.record()

In [17]:
iteration = first_iter
print(f"gaussians.pretrained_exposures: {gaussians.pretrained_exposures}")
a = gaussians.exposure_optimizer.param_groups[0]['lr']
b = gaussians.optimizer.param_groups[0]['lr']
print(f"gaussians.exposure_optimizer's lr: {a}")
print(f"gaussians.optimizer's lr: {b}")
print(f"gaussians.optimizer.param_groups[0]['name']: {gaussians.optimizer.param_groups[0]['name']}")

gaussians.pretrained_exposures: None
gaussians.exposure_optimizer's lr: 0.001
gaussians.optimizer's lr: 0.000812694320678711
gaussians.optimizer.param_groups[0]['name']: xyz


In [18]:
gaussians.update_learning_rate(iteration)

0.000812569577064852

In [19]:
print(f"gaussians.exposure_optimizer's lr: {gaussians.exposure_optimizer.param_groups[0]['lr']}")
print(f"gaussians.optimizer's lr: {gaussians.optimizer.param_groups[0]['lr']}")
print(f"gaussians.exposure_optimizer's lr's deviation: {a - gaussians.exposure_optimizer.param_groups[0]['lr']}")
print(f"gaussians.optimizer's lr's deviation: {b - gaussians.optimizer.param_groups[0]['lr']}")

gaussians.exposure_optimizer's lr: 0.00999923250108991
gaussians.optimizer's lr: 0.000812569577064852
gaussians.exposure_optimizer's lr's deviation: -0.008999232501089909
gaussians.optimizer's lr's deviation: 1.2474361385896634e-07


In [20]:
# Every 1000 its we increase the levels of SH up to a maximum degree
print(f"gaussians.active_sh_degree: {gaussians.active_sh_degree}, gaussians.max_sh_degree: {gaussians.max_sh_degree}")
if iteration % 1000 == 0:
    gaussians.oneupSHdegree()
print(f"gaussians.active_sh_degree after oneupSHdegree: {gaussians.active_sh_degree}")

gaussians.active_sh_degree: 0, gaussians.max_sh_degree: 3
gaussians.active_sh_degree after oneupSHdegree: 0


In [36]:
# Pick a random Camera
rand_idx = randint(0, len(viewpoint_indices) - 1)
print(f"rand_idx: {rand_idx}")
viewpoint_cam = viewpoint_stack.pop(rand_idx)
print(f"viewpoint_cam: {viewpoint_cam.uid}")
print(f"viewpoint_cam.colmap_id: {viewpoint_cam.colmap_id}")
print(f"viewpoint_cam.R: {viewpoint_cam.R}")
print(f"viewpoint_cam.T: {viewpoint_cam.T}")
print(f"viewpoint_cam.FoVx: {viewpoint_cam.FoVx}, viewpoint_cam.FoVy: {viewpoint_cam.FoVy}")
print(f"viewpoint_cam.image_name: {viewpoint_cam.image_name}")
vind = viewpoint_indices.pop(rand_idx)
print(f"vind: {vind}")

rand_idx: 20
viewpoint_cam: 20
viewpoint_cam.colmap_id: 1
viewpoint_cam.R: [[ 0.97323903  0.21118334  0.09059469]
 [-0.21487026  0.97608588  0.03297155]
 [-0.08146515 -0.0515553   0.99534189]]
viewpoint_cam.T: [ 0.56341554 -2.50232877  1.53516351]
viewpoint_cam.FoVx: 0.602981540771676, viewpoint_cam.FoVy: 1.0060968822522376
viewpoint_cam.image_name: 0028.jpg
vind: 20


In [38]:
# Render
print(f"args.debug_from: {args.debug_from}, iteration: {iteration}")
if (iteration - 1) == args.debug_from:
    pipe_args.debug = True
print(f"pipe_args.debug: {pipe_args.debug}")

args.debug_from: -1, iteration: 1
pipe_args.debug: False


In [41]:
print(f"opt_args.random_background: {opt_args.random_background}")
bg = torch.rand((3), device="cuda") if opt_args.random_background else background
print(f"bg: {bg}")

opt_args.random_background: False
bg: tensor([0., 0., 0.], device='cuda:0')


In [42]:
render_pkg = render(viewpoint_cam, gaussians, pipe_args, bg, use_trained_exp=model_args.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)
image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]
